# Anonymize a dataset with the Nucleus API

A full walkthrough of one anonymization: upload the file, follow the run, collect
the delivery and reverse it. A single CSV, `empleados.csv`, sitting next to this
notebook.

You need `requests` and `pandas`, and a Nucleus service token.


In [ ]:
import io
import json
import time
from pathlib import Path

import pandas as pd
import requests

BASE_URL = "https://api.dedomena.ai" 
TOKEN = "<your-service-token>"

CSV = Path("data/empleados.csv")
DATASET = CSV.stem          # 'empleados' — must match datasetName

session = requests.Session()
print(f"{BASE_URL} | dataset '{DATASET}'")


https://public-api-dev-587494419013.europe-southwest1.run.app | dataset 'empleados'


## 1. The starting data

Twelve employees with first name, surname, email, phone, national ID, date of
birth, postal code, salary and a free-text field that mentions other people.


In [2]:
original = pd.read_csv(CSV, sep=";")
print(original.shape)
original.head(3)


(12, 11)


,id_empleado,nombre,apellidos,email,telefono,dni,fecha_nacimiento,codigo_postal,departamento,salario,notas
0,EMP-000001,Valeria,Ordóñez Pastor,valeria.ordonez@ejemplo-ficticio.es,+34 611 002 001,12345678Z,1988-03-14,8001,Ingeniería,48200,Valeria Ordóñez pidió reducción de jornada por...
1,EMP-000002,Nicolás,Rebollo Antúnez,nicolas.rebollo@ejemplo-ficticio.es,+34 611 002 002,23456789D,1979-11-02,28014,Ingeniería,53750,Nicolás lleva el proyecto de migración. Report...
2,EMP-000003,Amparo,Cifuentes Gálvez,amparo.cifuentes@ejemplo-ficticio.es,+34 611 002 003,34567890X,1995-06-25,46007,Comercial,31400,Amparo cerró la cuenta de Talleres Berrocal. C...


## 2. The configuration

One method per column. **Columns that are not listed are delivered untouched** —
here `departamento`, which identifies nobody.

| Column | Method | Why |
|---|---|---|
| `id_empleado` | `pseudonym` with a pattern | new identifiers with the same shape, `EMP-000001` → `EMP-481902` |
| `nombre`, `apellidos` | `simulate` | believable names, not noise |
| `email` | `simulate` + `derive_from` | the address is built from the **already substituted** name, so the two match |
| `telefono` | `simulate` | reserved range, never somebody's real number |
| `dni` | `mask` | covered up, not replaced: it does not need to be valid |
| `fecha_nacimiento` | `date_shift` | shifts the date keeping the same offset for each employee |
| `codigo_postal` | `generalize` with a bucket of 1000 | the postcode is read as a number, so a bucket of 1000 leaves the province: `28014` → `28000-29000` |
| `salario` | `perturb` | 5% noise, the distribution is preserved |
| `notas` | `coding` | free text: it detects the people mentioned and replaces them with the **same** surrogate used in their own column |

A word on `generalize`, because it does two different things: on a numeric column
`param` is the **width of the bucket**, and on a text column it is the **minimum
number of times a value has to appear** before it is kept — everything rarer is
merged into `Others`. On a column where nearly every value is unique, that second
form empties the column instead of generalizing it, and the run stops rather than
deliver it. Twelve employees with twelve different postcodes is exactly that case,
so this one stays numeric.

Two details that save work: `token` and `assetId` can be omitted —the API takes
the first from the request token, and the second is resolved from the file
name—. The full catalogue of methods and their parameters is at
`GET /nucleus/anonymization_methods`.


In [ ]:
CONFIG = {
    "anonymizerName": "empleados-demo",
    "anonymizerDescription": "Example anonymization from the API",
    "anonymizerUseCase": "103",           # External Data Sharing
    "datasets": [
        {
            "datasetName": DATASET,
            "columns": {
                "id_empleado":      {"method": "pseudonym", "param": r"EMP-\d{6}"},
                "nombre":           {"method": "simulate", "param": "first_name"},
                "apellidos":        {"method": "simulate", "param": "last_names"},
                "email":            {"method": "simulate", "param": "email",
                                     "derive_from": ["nombre", "apellidos"]},
                "telefono":         {"method": "simulate", "param": "phone"},
                "dni":              {"method": "mask", "param": "opaque"},
                "fecha_nacimiento": {"method": "date_shift", "param": "id_empleado"},
                # a bucket of 1000 leaves the first two digits: it generalizes by province
                "codigo_postal":    {"method": "generalize", "param": 1000},
                "salario":          {"method": "perturb", "param": 0.05},
                "notas":            {"method": "coding", "param": "surrogates"},
            },
        }
    ],
}

print(json.dumps(CONFIG, indent=2, ensure_ascii=False)[:400], "...")


## 3. Launch the anonymization

`POST /nucleus/anonymize` takes four kinds of input, and exactly one per call:

- **`files`** — the files are attached, as here. csv, txt, tsv, json, jsonl,
  parquet, xlsx or avro.
- **`rows`** — the data in the body itself, as JSON. That is the next cell.
- **`collection_id`** — data already uploaded through axon, every asset in a
  collection.
- **`asset_id`** — a single dataset already uploaded.

The first two are tabular only. Images, documents and audio are anonymized
through the last two.

The `token` goes in the query string; the configuration and the file go as form
fields. It answers straight away with the `runId`: the job carries on behind the
scenes.


In [ ]:
with open(CSV, "rb") as fh:
    response = session.post(
        f"{BASE_URL}/nucleus/anonymize",
        params={"token": TOKEN},
        data={
            "configuration": json.dumps(CONFIG, ensure_ascii=False),
            "data_source": 232,        # 231 public domain | 232 own | 233 licensed
            # country and use_cases have defaults, no need to send them
        },
        files={"files": (CSV.name, fh, "text/csv")},
        timeout=120,
    )

response.raise_for_status()           # a 422 here says what the configuration is missing
run_id = response.json()["runId"]
print(run_id, response.json())


1e07f0d1-02f3-4789-8ea4-14b83626dc01 {'runId': '1e07f0d1-02f3-4789-8ea4-14b83626dc01', 'collectionId': None, 'status': 'processing', 'datasets': ['empleados']}


### The same call, without files

If you would rather not build a `multipart` request —the usual case from a
backend— the same route accepts **a JSON body**: the configuration is the body,
and the rows go inside each dataset under `rows`. What were form fields become
body fields: `dataSource`, and `country` and `useCases` if you need them.

The data travels whole in a single request and is held in memory, so this route
is for thousands of rows, not millions. Above that, attach the file or upload it
beforehand with axon and pass `collectionId`.


In [5]:
body = {
    **{k: v for k, v in CONFIG.items() if k != "datasets"},
    "dataSource": 232,
    "datasets": [
        {
            **CONFIG["datasets"][0],
            # The rows, as they are: a list of objects per dataset
            "rows": original.to_dict(orient="records"),
        }
    ],
}

r = session.post(f"{BASE_URL}/nucleus/anonymize",
                 params={"token": TOKEN}, json=body, timeout=120)
r.raise_for_status()
run_id_json = r.json()["runId"]
print(run_id_json)


961dc090-1b9c-4c87-b751-f8131b1fa8a1


## 4. Follow the run

`GET /nucleus/anonymize/runs/{run_id}` answers from the very first moment:
`10` in progress, `50` finished, `51` failed.


In [ ]:
def run_status(run_id):
    r = session.get(f"{BASE_URL}/nucleus/anonymize/runs/{run_id}",
                    params={"token": TOKEN}, timeout=60)
    r.raise_for_status()
    return r.json()


while True:
    info = run_status(run_id)
    print(info.get("status"), end=" ", flush=True)
    if str(info.get("status")) in {"50", "51"}:
        break
    time.sleep(15)

print()
assert str(info["status"]) == "50", f"the run failed: {info}"
info


10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 10 

## 5. Collect the delivery

The API does not serve the file: it signs the object in storage and returns a
temporary URL, which is the usual arrangement for large deliveries. It expires in
an hour unless you ask for something else (`expires_minutes`, up to a day).

The link grants read access to the data until it expires, so treat it as a
secret: do not paste it into a ticket or a chat.


In [ ]:
r = session.get(f"{BASE_URL}/nucleus/download/anonymized/{run_id}",
                params={"token": TOKEN, "expires_minutes": 30}, timeout=60)
r.raise_for_status()
delivery = r.json()

for d in delivery["datasets"]:
    print(d["name"], "->", d["assetId"])

# The delivered file uses ; and a BOM, so that Excel opens it properly
download = requests.get(delivery["datasets"][0]["url"], timeout=300)
download.raise_for_status()
anonymized = pd.read_csv(io.BytesIO(download.content), sep=";", encoding="utf-8-sig")
anonymized.head(3)


## 6. What happened to the data

Worth a look:

- the email matches the new name, not the old one;
- the same employee keeps the same identifier across all their rows;
- in `notas`, the people mentioned carry the surrogate assigned to them in their
  own column, not a different one;
- `departamento` is unchanged: it was not in the configuration.


In [ ]:
cols = ["id_empleado", "nombre", "apellidos", "email", "departamento"]
comparison = original[cols].join(anonymized[cols], lsuffix="_before", rsuffix="_after")
comparison.head(4)


In [ ]:
print("BEFORE:", original.loc[0, "notas"])
print()
print("AFTER :", anonymized.loc[0, "notas"])


## 7. Reverse it

`POST /nucleus/deanonymize` recovers the original values of the columns you ask
for, as long as the asset's mapping table is still kept.


In [ ]:
r = session.post(
    f"{BASE_URL}/nucleus/deanonymize",
    json={"token": TOKEN,
          "assetId": delivery["datasets"][0]["assetId"],
          "columns": ["nombre", "apellidos"]},
    timeout=120,
)
r.raise_for_status()
r.json()


## Where to go from here

**Data that is already uploaded.** Instead of `files`, one of the other two.
Nothing is uploaded again and the `assetId` is resolved from the asset name:

```python
data = {"configuration": json.dumps(CONFIG), "collection_id": "<id>"}
session.post(f"{BASE_URL}/nucleus/anonymize", params={"token": TOKEN}, data=data)
```

**Several related tables.** A `datasets` list with more than one entry, and
`references` so a foreign key gets the same surrogate as the entity in its source
table — it is written as `table.column`, so the whole configuration can be
drafted before uploading anything:

```python
"id_empleado": {"method": "pseudonym", "references": "empleados.id_empleado"}
```

**New data on top of a delivery already made.** `POST /nucleus/anonymize/increment`
with the previous `run_id` and only the new rows: it inherits that run's
configuration and its mappings, and does not touch what was already delivered.

**The catalogue of methods**, with the parameters each one accepts and which ones
apply to each asset type:


In [ ]:
r = session.get(f"{BASE_URL}/nucleus/anonymization_methods",
                params={"token": TOKEN}, timeout=60)
r.raise_for_status()
catalog = r.json()

for code, m in sorted(catalog["methods"].items()):
    print(f"{code:16} {m.get('param') or ''}")
